# Sentiment Analysis using SimpleRNN with Embedding

## Workflow:
1. **Tokenization**: Convert text → integers
2. **Padding**: Make all sequences same length
3. **Embedding**: Convert integers → dense vectors
4. **SimpleRNN**: Process sequence of vectors
5. **Prediction**: Output sentiment (0 or 1)

In [ ]:
# Sample text documents (examples for training)
docs = ['go india',
		'india india',
		'hip hip hurray',
		'jeetega bhai jeetega india jeetega',
		'bharat mata ki jai',
		'kohli kohli',
		'sachin sachin',
		'dhoni dhoni',
		'modi ji ki jai',
		'inquilab zindabad']

In [ ]:
# Step 1: Import Tokenizer - converts text to numbers
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer object (empty, will learn vocabulary later)
tokenizer = Tokenizer()

In [ ]:
import sys
if 'google.colab' in sys.modules:
  !pip install tensorflow

In [ ]:
# Step 2: Fit tokenizer on our documents
# This learns all unique words and assigns them IDs (word_index)
tokenizer.fit_on_texts(docs)

In [ ]:
# Check vocabulary size (unique words)
# word_index is a dict like {'go': 1, 'india': 2, 'hip': 3, ...}
len(tokenizer.word_index)

17

In [ ]:
# Step 3: Convert text → sequences of integers
# Example: 'go india' → [1, 2] (using word IDs)
sequences = tokenizer.texts_to_sequences(docs)
sequences

[[9, 1],
 [1, 1],
 [3, 3, 10],
 [2, 11, 2, 1, 2],
 [12, 13, 4, 5],
 [6, 6],
 [7, 7],
 [8, 8],
 [14, 15, 4, 5],
 [16, 17]]

In [ ]:
# Step 4: Pad sequences to same length
# Different sequences have different lengths, so we pad with zeros (padding='post' = add zeros at end)
from tensorflow.keras.utils import pad_sequences
sequences = pad_sequences(sequences, padding='post')
sequences

array([[ 9,  1,  0,  0,  0],
       [ 1,  1,  0,  0,  0],
       [ 3,  3, 10,  0,  0],
       [ 2, 11,  2,  1,  2],
       [12, 13,  4,  5,  0],
       [ 6,  6,  0,  0,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [14, 15,  4,  5,  0],
       [16, 17,  0,  0,  0]], dtype=int32)

In [ ]:
# Step 5: Create model with Embedding layer
# Embedding: converts integer IDs → dense vectors (17 vocab words, dimension=2)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding # converts integers to dense vectors

model = Sequential()
# input_dim=17 (vocab size), output_dim=2 (embedding dimension), input_length=5 (sequence length)
model.add(Embedding(17, output_dim=2, input_length=5))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
# optimizer='adam', loss function, metric='accuracy'
model.compile('adam', 'accuracy')

In [ ]:
# Get predictions from the model
# Output shape: (10, 5, 2) = 10 samples, 5 timesteps, 2 dimensions
pred = model.predict(sequences)
print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 785ms/step
[[[ 0.02947644  0.04597727]
  [-0.00900982  0.04446869]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]]

 [[-0.00900982  0.04446869]
  [-0.00900982  0.04446869]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]]

 [[ 0.00956174  0.01864712]
  [ 0.00956174  0.01864712]
  [-0.02919968  0.0244643 ]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]]

 [[ 0.01163341  0.00521201]
  [-0.04981145  0.0095141 ]
  [ 0.01163341  0.00521201]
  [-0.00900982  0.04446869]
  [ 0.01163341  0.00521201]]

 [[-0.02868345  0.04397166]
  [ 0.04890165 -0.04316213]
  [ 0.01629485 -0.00491484]
  [ 0.04797794 -0.04882212]
  [ 0.00369782  0.00580772]]

 [[ 0.04332213  0.00651   ]
  [ 0.04332213  0.00651   ]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.00580772]]

 [[-0.01274496  0.03013242]
  [-0.01274496  0.03013242]
  [ 0.00369782  0.00580772]
  [ 0.00369782  0.0058077

In [ ]:
# ============= PART 2: Real Sentiment Analysis (IMDB Dataset) =============
# Import all required libraries
from tensorflow.keras.datasets import imdb  # Sentiment dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Embedding, Flatten

In [ ]:
# Load IMDB dataset (movie reviews: 1=positive, 0=negative)
(X_train, y_train), (X_test, y_test) = imdb.load_data()

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Pad all sequences to fixed length (50) for consistency
# This ensures RNN gets fixed-size input
X_train = pad_sequences(X_train, padding='post', maxlen=50)
X_test = pad_sequences(X_test, padding='post', maxlen=50)

In [ ]:
# Check shape: (samples, sequence_length)
X_train.shape

(25000, 50)

In [ ]:
# Build SimpleRNN model for sentiment classification
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, Embedding

model = Sequential()
# Layer 1: Embedding (10000 vocab, 2-dim embeddings)
model.add(Embedding(10000, 2))
# Layer 2: SimpleRNN (32 units, processes sequence)
model.add(SimpleRNN(32, return_sequences=False))
# Layer 3: Dense output (sigmoid for binary classification: positive/negative)
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile and train the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
# Train for 10 epochs
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - acc: 0.8869 - loss: 0.2894 - val_acc: 0.7773 - val_loss: 0.5254
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - acc: 0.8955 - loss: 0.2667 - val_acc: 0.7563 - val_loss: 0.5803
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - acc: 0.9085 - loss: 0.2469 - val_acc: 0.7678 - val_loss: 0.5924
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.9117 - loss: 0.2328 - val_acc: 0.7595 - val_loss: 0.6472
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.9238 - loss: 0.2126 - val_acc: 0.7516 - val_loss: 0.6744
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - acc: 0.9229 - loss: 0.2091 - val_acc: 0.7444 - val_loss: 0.7162
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.9287 - loss: 0.1971 - val_acc: 0.7393 - val_loss: 0.8314
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.9378 - loss: 0.1788 - val_acc: 0.7362 - val_loss: 0.7936
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step -